# Point-cloud autoencoder on IFCNet: anomaly detection + similarity
Same task as `autoencoder.ipynb`, but on the 3D geometry of each IFC element (IFCNetCore point clouds) instead of 2D renders.

In [ ]:
from pathlib import Path
import random, time, json
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ROOT = Path("../../data/ply")      # data/ply/<IfcClass>/<train|test>/<id>.ply
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
N_POINTS = 1024                    # points per object fed to the model (from 2048)
CACHE = ROOT / f"ifcnet_pc{N_POINTS}_cache.pt"

## 1. The data
Each object is one PLY file with 2,048 surface points. We take a fixed random subset of 1,024 points, move the centre to the origin and scale it into the unit sphere, so only the **shape** matters, not position or size. Train/test come from IFCNet's folders; validation is split from train by object.

In [ ]:
def load_points(path, n=N_POINTS, rng=None):
    pts = np.asarray(trimesh.load(path).vertices, dtype=np.float32)
    rng = rng or np.random.default_rng(0)
    idx = rng.choice(len(pts), n, replace=len(pts) < n)   # fixed subset of n points
    pts = pts[idx]
    pts -= pts.mean(0)                                      # centre at origin
    pts /= np.linalg.norm(pts, axis=1).max() + 1e-9         # into unit sphere
    return pts

if CACHE.exists():
    d = torch.load(CACHE)
    print("loaded", CACHE)
else:
    files = sorted(ROOT.glob("*/*/*.ply"))
    classes = sorted({f.parts[-3] for f in files})
    rng = np.random.default_rng(SEED)
    data = {"train": ([], [], []), "test": ([], [], [])}
    t0 = time.time()
    for f in files:
        cls, split, oid = f.parts[-3], f.parts[-2], f.stem
        X, y, ids = data[split]
        X.append(load_points(f, rng=rng)); y.append(classes.index(cls)); ids.append(oid)
    print(f"read {len(files)} files in {time.time()-t0:.0f}s")
    d = {"classes": classes}
    for split, (X, y, ids) in data.items():
        d[f"X_{split}"] = torch.tensor(np.stack(X))      # [N, 1024, 3]
        d[f"y_{split}"] = torch.tensor(y)
        d[f"ids_{split}"] = ids
    torch.save(d, CACHE)
    print("saved", CACHE)

classes = d["classes"]
# validation split by object (each object is one file, so no view leakage)
n = len(d["X_train"]); perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
val_idx, tr_idx = perm[: n // 7], perm[n // 7 :]
X_train, y_train = d["X_train"][tr_idx], d["y_train"][tr_idx]
X_val,   y_val   = d["X_train"][val_idx], d["y_train"][val_idx]
X_test,  y_test  = d["X_test"], d["y_test"]
print(len(classes), "classes")
print("train:", tuple(X_train.shape), "val:", tuple(X_val.shape), "test:", tuple(X_test.shape))

### A first look: one object per class

In [ ]:
fig = plt.figure(figsize=(16, 7))
for c in range(len(classes)):
    i = int((y_train == c).nonzero()[0])
    p = X_train[i].numpy()
    ax = fig.add_subplot(2, 10, c + 1, projection="3d")
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=1, c=p[:, 2], cmap="viridis")
    ax.set_title(classes[c][3:], fontsize=7); ax.set_axis_off()
    ax.set_box_aspect((1, 1, 1))
plt.suptitle(f"IFCNet point clouds: one training object per class ({N_POINTS} points, unit sphere)")
plt.tight_layout(); plt.savefig(FIG / "3d_01_samples.png", dpi=150); plt.show()

counts = torch.bincount(y_train, minlength=len(classes))
print({classes[c][3:]: int(counts[c]) for c in range(len(classes))})